# 12.10 - AI-Powered UX

**Module 12 - Production Deployment** | Netsetos GenAI Engineering

This notebook works through AI-Powered UX hands-on: Streaming collapses the perceived wait to time-to-first-token; Match the loading indicator to the wait, and prefer a skeleton; Optimistic UI: echo the action first, reconcile or roll back; Render reasoning as a collapsible disclosure, not the answer; Attach citations to claims and cap confidence on the weakest; Map every error to a friendly message plus recovery, and degrade.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This lesson's cells are all pure-Python models with no API calls, so setup is minimal: only the optional Anthropic install (for the illustrative streaming stack later) and a note that the numbers are fixed constants, not random draws.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

Environment prep only. There is nothing to import here - the install line is commented out and the second line is just a reminder that every value in the notebook is a hard-coded constant, so the outputs are fully reproducible with no seed needed.

**How the code works - step by step**
- **Commented `pip install anthropic`** - only needed if you later run the illustrative streaming stack; the seven core cells need nothing.
- **Reproducibility comment** - flags that the lesson uses fixed values throughout, so every run prints the same output.

**In one line:** no imports, no key, no randomness - just fixed constants.

**What you'll see:** (no output - environment prep)

## 1 - Streaming collapses the perceived wait to time-to-first-token

The single most valuable AI-product UX decision: stream tokens instead of holding the whole answer. This cell races two front-ends over the **same** 3-second generation and measures the one thing that changes - the perceived wait - while the total time stays identical.

In [ ]:
# The backend is identical. Only the UX changes the PERCEIVED wait.
TOTAL_MS = 3000     # same generation time either way
TOKENS = 150
TTFT_MS = 200       # time to FIRST token when streaming

# BLANK: hold everything, reveal the whole answer at the end.
blank_first_content = TOTAL_MS          # user stares at nothing until it is all done
# STREAMING: first token at TTFT, then the user reads while the model keeps generating.
stream_first_content = TTFT_MS
per_token = (TOTAL_MS - TTFT_MS) / TOKENS

print("Same request, same model, {}ms to generate {} tokens. Two front-ends:".format(TOTAL_MS, TOKENS))
print()
print("  BLANK WAIT:  nothing on screen for {}ms, then the whole answer appears at once.".format(blank_first_content))
print("     -> the user stares at a blank screen for {}ms".format(blank_first_content))
print("  STREAMING:   first token at {}ms, then ~1 token every {:.0f}ms while the user reads.".format(TTFT_MS, per_token))
print("     -> the user starts reading after just {}ms".format(stream_first_content))
print()
speedup = blank_first_content / stream_first_content
print("Perceived wait (time to first content): {}ms -> {}ms = {:.0f}x lower.".format(blank_first_content, stream_first_content, speedup))
print("Total time is UNCHANGED ({}ms). Streaming is a UX lever, not a backend one:".format(TOTAL_MS))
print("the user reads while the model thinks, so the wait feels near-zero. Always show a stop button.")

# Output:
# Same request, same model, 3000ms to generate 150 tokens. Two front-ends:
#
#   BLANK WAIT:  nothing on screen for 3000ms, then the whole answer appears at once.
#      -> the user stares at a blank screen for 3000ms
#   STREAMING:   first token at 200ms, then ~1 token every 19ms while the user reads.
#      -> the user starts reading after just 200ms
#
# Perceived wait (time to first content): 3000ms -> 200ms = 15x lower.
# Total time is UNCHANGED (3000ms). Streaming is a UX lever, not a backend one:
# the user reads while the model thinks, so the wait feels near-zero. Always show a stop button.

**Explanation**

This is a measurement harness, not a model call. It fixes the backend (3000ms to make 150 tokens) and computes when the user first sees content under two front-ends: a blank wait reveals everything at the end, while streaming shows the first token at TTFT. The whole point is that total time is a constant and only the front-end moves the perceived wait.

**How the code works - step by step**
- **Constants** - `TOTAL_MS=3000`, `TOKENS=150`, `TTFT_MS=200` fix the identical backend for both paths.
- **`blank_first_content`** - set to the full `TOTAL_MS`: the user sees nothing until generation completes.
- **`stream_first_content`** - set to `TTFT_MS`: the first token lands early and the user reads while the rest streams.
- **`per_token`** - the remaining time spread over the tokens, ~19ms each, printed as the streaming cadence.
- **`speedup`** - the ratio of the two perceived waits, showing the many-fold drop.

**In one line:** same total time, but perceived wait falls from the whole generation to just the TTFT.

**What you'll see:** A two-front-end comparison: blank wait shows nothing for 3000ms, streaming shows the first token at 200ms with ~1 token every 19ms, so perceived wait drops 3000ms -> 200ms = 15x lower while total time stays 3000ms.

## 2 - Match the loading indicator to the wait, and prefer a skeleton

Before the first token arrives you still have a wait to fill, and not all waits deserve the same indicator. This cell encodes Nielsen's three response-time limits as a selector, then quantifies why a skeleton beats a bare spinner.

In [ ]:
# Match the indicator to the wait. Nielsen's three response-time limits (0.1s / 1s / 10s).
def indicator(expected_ms):
    if expected_ms < 100:    return "nothing (feels instant)"
    if expected_ms < 1000:   return "a subtle hint (keep the flow, no spinner needed)"
    if expected_ms < 10000:  return "a SKELETON of the coming content (hold attention)"
    return "PHASED messages + an estimate + a Cancel button"

print("The right loading indicator by expected wait:")
for ms in [80, 600, 4000, 25000]:
    print("  {:>6}ms -> {}".format(ms, indicator(ms)))
print()

# Skeleton vs spinner: a skeleton sets expectations, so the SAME wait FEELS ~40% shorter.
actual_ms = 4000
SKELETON_FACTOR = 0.6    # perceived wait is ~60% of actual with a skeleton (~40% cut)
spinner_perceived = actual_ms
skeleton_perceived = int(actual_ms * SKELETON_FACTOR)
print("Skeleton vs spinner for the same {}ms wait:".format(actual_ms))
print("  blank panel + spinner: feels like {}ms (opaque, reads as broken past ~1s)".format(spinner_perceived))
print("  skeleton screen:       feels like {}ms (~40% lower - it shows the shape of what is coming)".format(skeleton_perceived))
print()
print("A spinner past ~1s reads as stuck; a skeleton (a few grey lines at decreasing widths) reassures.")

# Output:
# The right loading indicator by expected wait:
#       80ms -> nothing (feels instant)
#      600ms -> a subtle hint (keep the flow, no spinner needed)
#     4000ms -> a SKELETON of the coming content (hold attention)
#    25000ms -> PHASED messages + an estimate + a Cancel button
#
# Skeleton vs spinner for the same 4000ms wait:
#   blank panel + spinner: feels like 4000ms (opaque, reads as broken past ~1s)
#   skeleton screen:       feels like 2400ms (~40% lower - it shows the shape of what is coming)
#
# A spinner past ~1s reads as stuck; a skeleton (a few grey lines at decreasing widths) reassures.

**Explanation**

A small decision function plus one perceived-time calculation. `indicator` maps an expected wait to the right UI by Nielsen's 0.1s / 1s / 10s thresholds, and the second block models the finding that a skeleton makes the same wait feel about 40 percent shorter than a spinner. Both are UX rules expressed as plain arithmetic.

**How the code works - step by step**
- **`indicator(expected_ms)`** - four-branch selector: nothing under 100ms, a subtle hint under 1000ms, a skeleton under 10000ms, phased-plus-cancel beyond.
- **Selector loop** - runs `indicator` over `[80, 600, 4000, 25000]` to show one example per tier.
- **`SKELETON_FACTOR=0.6`** - the perceived-wait multiplier: a skeleton feels ~60% of the actual wait.
- **`spinner_perceived` vs `skeleton_perceived`** - the same 4000ms wait rendered two ways, the skeleton ~40% lower.

**In one line:** pick the indicator by wait length, and a skeleton makes the wait feel shorter than a spinner.

**What you'll see:** A per-threshold table (80ms->nothing, 600ms->subtle, 4000ms->skeleton, 25000ms->phased+cancel) then a skeleton-vs-spinner comparison where the same 4000ms wait feels like 4000ms with a spinner but 2400ms with a skeleton.

## 3 - Optimistic UI: echo the action first, reconcile or roll back

Streaming handles the model's response; optimistic UI handles the user's own action. This cell runs both the success and failure timelines of a send so you can see the message appear at t=0 - before any round-trip - and what the failure path must clean up.

In [ ]:
# Optimistic UI: echo the user's action BEFORE the round-trip, then reconcile or roll back.
SERVER_MS = 800    # how long the real round-trip takes

def send(message, server_ok):
    log = []
    log.append("  t=0ms    user hits send -> message appears in the thread INSTANTLY (optimistic)")
    log.append("  t=0ms    perceived send-latency for the user: 0ms")
    log.append("  t={}ms  server responds...".format(SERVER_MS))
    if server_ok:
        log.append("  t={}ms  RECONCILE: server confirmed -> keep the message, mark it delivered".format(SERVER_MS))
    else:
        log.append("  t={}ms  ROLLBACK: server failed -> remove the message, restore the input, show a clear error".format(SERVER_MS))
    return log

print("Optimistic send that SUCCEEDS:")
for line in send("what is MCP?", True): print(line)
print()
print("Optimistic send that FAILS:")
for line in send("what is MCP?", False): print(line)
print()
print("Naive fetch-then-render would show nothing for {}ms ('did my message even send?').".format(SERVER_MS))
print("Optimistic UI makes the send feel instant; the ONLY cost is you must handle the rollback path.")

# Output:
# Optimistic send that SUCCEEDS:
#   t=0ms    user hits send -> message appears in the thread INSTANTLY (optimistic)
#   t=0ms    perceived send-latency for the user: 0ms
#   t=800ms  server responds...
#   t=800ms  RECONCILE: server confirmed -> keep the message, mark it delivered
#
# Optimistic send that FAILS:
#   t=0ms    user hits send -> message appears in the thread INSTANTLY (optimistic)
#   t=0ms    perceived send-latency for the user: 0ms
#   t=800ms  server responds...
#   t=800ms  ROLLBACK: server failed -> remove the message, restore the input, show a clear error
#
# Naive fetch-then-render would show nothing for 800ms ('did my message even send?').
# Optimistic UI makes the send feel instant; the ONLY cost is you must handle the rollback path.

**Explanation**

A tiny timeline simulator. `send` builds a log of what the user sees over time for one message, branching on whether the server call succeeds: on success it reconciles and keeps the message, on failure it rolls back and restores the input. It is showing the two paths side by side, not calling any real server.

**How the code works - step by step**
- **`SERVER_MS=800`** - the real round-trip duration the optimistic echo hides.
- **`send(message, server_ok)`** - appends timeline lines: at t=0 the message appears instantly with zero perceived latency, at t=800ms the server responds.
- **Success branch** - RECONCILE: keep the message, mark it delivered.
- **Failure branch** - ROLLBACK: remove the message, restore the input, show a clear error.
- **Two calls** - one with `server_ok=True`, one `False`, to print both timelines.

**In one line:** show the message at t=0, then keep it on success or undo it on failure.

**What you'll see:** Two timelines for the same send: both show the message appearing instantly at t=0 with 0ms perceived latency; the success path reconciles at 800ms (keep, delivered), the failure path rolls back at 800ms (remove, restore input, error).

## 4 - Render reasoning as a collapsible disclosure, not the answer

Reasoning models emit a long chain-of-thought before a short answer, and how you render it makes or breaks the experience. This cell compares three renderings and the timing tradeoff to land on the one pattern that works.

In [ ]:
# A thinking model emits a long reasoning trace, then a short answer. Render it THREE ways.
reasoning_tokens = 620
answer_tokens = 90

options = {
    "dump as answer":       {"shown": reasoning_tokens + answer_tokens, "verdict": "BAD: user mistakes the chain-of-thought for the answer"},
    "hidden entirely":      {"shown": answer_tokens, "verdict": "power users lose the reasoning they wanted to inspect"},
    "collapsible, default closed": {"shown": answer_tokens, "verdict": "GOOD: answer up front, reasoning one click away"},
}
print("Rendering {} reasoning tokens + a {}-token answer:".format(reasoning_tokens, answer_tokens))
for name, o in options.items():
    print("  {:<28} user sees {:>3} tokens first  -> {}".format(name, o["shown"], o["verdict"]))
print()

# The silence-tax vs premature-commitment tradeoff.
print("Timing tradeoff:")
print("  wait for all {} reasoning tokens before showing anything -> a long BLANK (the silence tax)".format(reasoning_tokens))
print("  stream the answer with a live, collapsible 'Thinking' panel -> motion immediately, no confusion")
print()
print("Pattern: put reasoning in a collapsible 'Thinking' disclosure, default CLOSED, above the answer.")

# Output:
# Rendering 620 reasoning tokens + a 90-token answer:
#   dump as answer               user sees 710 tokens first  -> BAD: user mistakes the chain-of-thought for the answer
#   hidden entirely              user sees  90 tokens first  -> power users lose the reasoning they wanted to inspect
#   collapsible, default closed  user sees  90 tokens first  -> GOOD: answer up front, reasoning one click away
#
# Timing tradeoff:
#   wait for all 620 reasoning tokens before showing anything -> a long BLANK (the silence tax)
#   stream the answer with a live, collapsible 'Thinking' panel -> motion immediately, no confusion
#
# Pattern: put reasoning in a collapsible 'Thinking' disclosure, default CLOSED, above the answer.

**Explanation**

A comparison table over rendering choices. It fixes a 620-token reasoning trace plus a 90-token answer and, for each rendering option, records how many tokens the user sees first and a verdict - exposing that dumping the trace as the answer is the trap and a collapsible default-closed disclosure is the fix. The second block contrasts waiting (a silence tax) against streaming.

**How the code works - step by step**
- **`reasoning_tokens=620`, `answer_tokens=90`** - a long trace and a short answer.
- **`options` dict** - three renderings, each with the tokens shown first and a verdict: dump-as-answer (bad, 710 tokens first), hidden (loses power users), collapsible-default-closed (good, answer first).
- **Options loop** - prints each rendering with its shown-token count and verdict.
- **Timing tradeoff block** - waiting for all 620 reasoning tokens is a blank silence tax; streaming the answer with a live 'Thinking' panel shows motion immediately.

**In one line:** answer up front, reasoning one click away in a default-closed 'Thinking' panel.

**What you'll see:** A three-row rendering comparison (dump-as-answer shows 710 tokens first and is BAD; hidden and collapsible both show 90 first, only collapsible is GOOD) plus a timing note that waiting for all reasoning is a silence tax while streaming avoids confusion.

## 5 - Attach citations to claims and cap confidence on the weakest

The hardest AI UX problem is honesty about uncertainty: one unsourced claim styled like a fact quietly kills trust. This cell scores an answer claim by claim and derives a confidence tier that any unsupported claim drags down.

In [ ]:
# Attach citations to CLAIMS, not paragraphs. The confidence tier follows the support.
claims = [
    {"text": "MCP is an open standard", "source": "Anthropic docs", "relevance": 0.95},
    {"text": "It uses JSON-RPC over stdio or HTTP", "source": "MCP spec", "relevance": 0.88},
    {"text": "It was first shipped in 2019", "source": None, "relevance": 0.0},   # unsupported, styled like a fact
]
supported = [c for c in claims if c["source"]]
coverage = len(supported) / len(claims)
min_rel = min((c["relevance"] for c in supported), default=0.0)

def tier(coverage, min_rel):
    if coverage < 1.0:      return "LOW - verify this answer"   # ANY unsupported claim caps it
    if min_rel >= 0.9:      return "HIGH"
    if min_rel >= 0.7:      return "MODERATE"
    return "LOW - verify this answer"

print("Answer, claim by claim:")
for c in claims:
    if c["source"]:
        print("  [cited]      {:<38} {} ({:.0f}% relevant)".format(c["text"], c["source"], c["relevance"] * 100))
    else:
        print("  [UNSOURCED]  {:<38} <- styled like a fact, but no source".format(c["text"]))
print()
t = tier(coverage, min_rel)
print("Citation coverage: {}/{} claims -> confidence tier: {}".format(len(supported), len(claims), t))
if "LOW" in t:
    print("Show a 'verify this' disclaimer and flag the unsourced claim; do NOT hide the uncertainty.")
print("One unsourced claim styled like a sourced one is what erodes trust - honest disclosure builds it.")

# Output:
# Answer, claim by claim:
#   [cited]      MCP is an open standard                Anthropic docs (95% relevant)
#   [cited]      It uses JSON-RPC over stdio or HTTP    MCP spec (88% relevant)
#   [UNSOURCED]  It was first shipped in 2019           <- styled like a fact, but no source
#
# Citation coverage: 2/3 claims -> confidence tier: LOW - verify this answer
# Show a 'verify this' disclaimer and flag the unsourced claim; do NOT hide the uncertainty.
# One unsourced claim styled like a sourced one is what erodes trust - honest disclosure builds it.

**Explanation**

A scoring model over individual claims. It attaches a source and relevance to each claim, computes citation coverage, and runs a `tier` function whose first rule caps the answer at LOW the moment any claim is unsourced - the mechanism behind 'verify this'. This is claim-level, not paragraph-level, honesty encoded as a function.

**How the code works - step by step**
- **`claims` list** - three claims, two with a source and relevance, one with `source=None` (the fact-styled but unsupported one).
- **`supported` / `coverage`** - filters cited claims and computes the fraction that are sourced.
- **`min_rel`** - the weakest relevance among supported claims.
- **`tier(coverage, min_rel)`** - returns LOW if coverage < 1.0 (any unsourced claim caps it), else HIGH/MODERATE/LOW by relevance.
- **Claim loop + verdict** - prints each claim as [cited] or [UNSOURCED], then the tier and, if LOW, the 'verify this' + flag instruction.

**In one line:** score each claim, and one unsourced claim drops the whole answer to LOW - verify this.

**What you'll see:** A claim-by-claim listing (two cited with relevance, one flagged [UNSOURCED]), then coverage 2/3 -> confidence tier LOW - verify this, with instructions to show the disclaimer and flag the unsourced claim rather than hide it.

## 6 - Map every error to a friendly message plus recovery, and degrade

Everything breaks eventually, and no user should ever see a raw '500 Internal Server Error'. This cell maps each failure code to a friendly title and a recovery action, then models graceful degradation to a cached answer.

In [ ]:
# No user ever sees a raw code. Map every failure to a friendly message + a recovery action.
ERRORS = {
    401: {"title": "Session expired",     "action": "redirect to log in"},
    402: {"title": "Usage limit reached", "action": "show upgrade options"},
    429: {"title": "Too many requests",   "action": "auto-retry in 5s (countdown)"},
    500: {"title": "Our AI hit a snag",   "action": "auto-retry in 3s, else fall back to a cached answer"},
    503: {"title": "High demand",         "action": "queue the request, auto-retry in 10s"},
    "timeout": {"title": "Taking a while","action": "offer 'simplify the question'"},
    "network": {"title": "You are offline","action": "auto-retry on reconnect"},
}
def show(code):
    e = ERRORS.get(code, ERRORS[500])
    return "{:<8} -> \"{}\"  ({})".format(str(code), e["title"], e["action"])

print("Every error becomes a friendly card with a recovery action:")
for code in [401, 402, 429, 500, 503, "timeout", "network"]:
    print("  " + show(code))
print()

# Graceful degradation: a cached/smaller answer beats a blank error.
cache_has_answer = True
print("Graceful degradation on a 500 (mid-stream):")
if cache_has_answer:
    print("  keep the partial streamed text, then serve a cached answer marked 'showing a saved response'")
else:
    print("  show the friendly retry card, never a raw 500")
print("A degraded but honest answer beats a blank 'Internal Server Error' every time.")

# Output:
# Every error becomes a friendly card with a recovery action:
#   401      -> "Session expired"  (redirect to log in)
#   402      -> "Usage limit reached"  (show upgrade options)
#   429      -> "Too many requests"  (auto-retry in 5s (countdown))
#   500      -> "Our AI hit a snag"  (auto-retry in 3s, else fall back to a cached answer)
#   503      -> "High demand"  (queue the request, auto-retry in 10s)
#   timeout  -> "Taking a while"  (offer 'simplify the question')
#   network  -> "You are offline"  (auto-retry on reconnect)
#
# Graceful degradation on a 500 (mid-stream):
#   keep the partial streamed text, then serve a cached answer marked 'showing a saved response'
# A degraded but honest answer beats a blank 'Internal Server Error' every time.

**Explanation**

A lookup table plus a degradation branch. `ERRORS` maps HTTP codes and named failures to a friendly title and the right recovery (auto-retry transient, redirect auth/billing, simplify on timeout), and `show` formats them - transient errors often heal with no user action. The final block shows the AI-specific move: fall back to a cached answer rather than a blank error.

**How the code works - step by step**
- **`ERRORS` dict** - maps 401/402/429/500/503/timeout/network to a `title` and `action` (log in, upgrade, auto-retry with countdown, queue, simplify, reconnect).
- **`show(code)`** - looks up a code (defaulting to 500) and formats it as a friendly card line.
- **Error loop** - runs `show` over every code to print the full recovery map.
- **Graceful-degradation block** - with `cache_has_answer=True`, keep the partial streamed text and serve a cached answer marked 'showing a saved response'.

**In one line:** turn every raw code into a friendly card with a recovery action, and degrade rather than blank.

**What you'll see:** A friendly card per error (401->Session expired, 429->Too many requests auto-retry, 500->Our AI hit a snag with cached fallback, etc.) then a degradation line: on a mid-stream 500, keep the partial text and serve a cached 'saved response'.

## 7 - Confirm the risky actions and capture thumbs feedback

The last mile has two closing patterns that connect the UX back to everything before it. This cell runs a confirm-before-you-act gate over a set of actions, then aggregates thumbs votes - linked to trace ids - into a rolling quality signal.

In [ ]:
# TWO closing patterns. (1) Confirm-before-you-act on risky actions; auto-run the safe ones.
def needs_confirm(action):
    reasons = []
    if action["destructive"]:   reasons.append("destructive")
    if action["irreversible"]:  reasons.append("irreversible")
    if action["cost_usd"] > 50: reasons.append("high-cost")
    if action["confidence"] < 0.6: reasons.append("low-confidence")
    return reasons

actions = [
    {"name": "answer an FAQ",        "destructive": False, "irreversible": False, "cost_usd": 0,   "confidence": 0.95},
    {"name": "read a document",      "destructive": False, "irreversible": False, "cost_usd": 0,   "confidence": 0.9},
    {"name": "delete 40 records",    "destructive": True,  "irreversible": True,  "cost_usd": 0,   "confidence": 0.8},
    {"name": "send a customer email","destructive": False, "irreversible": True,  "cost_usd": 0,   "confidence": 0.7},
    {"name": "place a $500 order",   "destructive": False, "irreversible": True,  "cost_usd": 500, "confidence": 0.85},
]
print("Confirm-before-you-act gate:")
for a in actions:
    reasons = needs_confirm(a)
    verdict = "CONFIRM ({})".format(", ".join(reasons)) if reasons else "auto-run"
    print("  {:<24} -> {}".format(a["name"], verdict))
print("  When the model is genuinely unsure, a short clarifying question beats a confident wrong action.")
print()

# (2) The feedback loop: thumbs up/down linked to a TRACE id, aggregated into a rolling quality signal.
votes = [("trace-01", 1), ("trace-02", 1), ("trace-03", 0), ("trace-04", 1), ("trace-05", 0), ("trace-06", 1)]
print("Feedback stream (each vote linked to its trace for the 12.8 eval loop):")
running = 0
for i, (trace, v) in enumerate(votes, 1):
    running += v
    print("  {}  thumbs-{:<4} rolling up-rate {:.0%}".format(trace, "up" if v else "down", running / i))
print("These votes are where production quality data is born - they feed online eval (12.8) and the eval set (14.4).")

# Output:
# Confirm-before-you-act gate:
#   answer an FAQ            -> auto-run
#   read a document          -> auto-run
#   delete 40 records        -> CONFIRM (destructive, irreversible)
#   send a customer email    -> CONFIRM (irreversible)
#   place a $500 order       -> CONFIRM (irreversible, high-cost)
#   When the model is genuinely unsure, a short clarifying question beats a confident wrong action.
#
# Feedback stream (each vote linked to its trace for the 12.8 eval loop):
#   trace-01  thumbs-up   rolling up-rate 100%
#   trace-02  thumbs-up   rolling up-rate 100%
#   trace-03  thumbs-down rolling up-rate 67%
#   trace-04  thumbs-up   rolling up-rate 75%
#   trace-05  thumbs-down rolling up-rate 60%
#   trace-06  thumbs-up   rolling up-rate 67%
# These votes are where production quality data is born - they feed online eval (12.8) and the eval set (14.4).

**Explanation**

Two small models in one cell. `needs_confirm` inspects an action's risk flags and returns the reasons it needs a human confirm (destructive, irreversible, high-cost, low-confidence), while the feedback block folds a stream of thumbs votes into a running up-rate keyed to trace ids. The first is the HITL gate; the second is where production quality data is born.

**How the code works - step by step**
- **`needs_confirm(action)`** - collects risk reasons: destructive, irreversible, `cost_usd > 50`, or `confidence < 0.6`; empty means safe to auto-run.
- **`actions` list** - five actions from an FAQ answer to deleting 40 records to a $500 order.
- **Gate loop** - prints each action as auto-run or CONFIRM with its reasons, plus the clarifying-question rule when unsure.
- **`votes` list + rolling loop** - each thumbs vote is tied to a trace id and folded into a running up-rate.

**In one line:** confirm the risky actions, and turn each trace-linked thumbs vote into a live quality signal.

**What you'll see:** A confirm gate (FAQ and read auto-run; delete/email/order require CONFIRM with reasons) then a feedback stream of six trace-linked votes with the rolling up-rate moving 100% -> 67% -> 75% -> 60% -> 67%, noted as feeding the 12.8 online eval and 14.4 eval set.

Together these seven models are the last mile of a production AI system: streaming and skeletons make the same latency feel fast, optimistic UI makes the user's own action feel instant, reasoning disclosure and claim-level citations make the model honest, friendly error mapping keeps it trustworthy when things break, and the confirm gate plus thumbs feedback close the loop back to your evals. Each cell is a tiny keyless model that isolates one UX decision so you can see exactly what changes the feel of a product without changing the backend. From here the cost budget behind these latencies is Module 13, and the eval set seeded by the thumbs feedback is built in Lesson 14.4.